# Panel DID simulation with true value (genriesz)

We implement DID as **ATT on** the differenced outcome

\[
\Delta Y = Y_1 - Y_0,
\]

where:

- `Y0` is the pre-period outcome,
- `Y1` is the post-period outcome,
- the same units are observed in both periods (panel),
- `D` is a binary treatment indicator (treatment happens in the post period).

With a standard panel DID setup:

\[
Y_{0} = \mu(Z) + u + \varepsilon_0, \qquad
Y_{1} = \mu(Z) + \text{trend}(Z) + u + \tau D + \varepsilon_1,
\]

the DID effect equals the constant treatment effect \(\tau\), provided the
parallel trends condition holds after conditioning on `Z`.

This notebook:

1. simulates a large population to compute an approximate "true" DID effect,
2. samples a dataset and calls `genriesz.grr_did(X, Y0=..., Y1=...)`.


In [ ]:
import numpy as np

from genriesz import (
    grr_did,
    SquaredGenerator,
    PolynomialBasis,
    TreatmentInteractionBasis,
)

rng = np.random.default_rng(0)


## DGP

In [ ]:
def draw_panel(n: int, d_z: int, tau: float, seed: int = 0):
    rng = np.random.default_rng(seed)
    Z = rng.normal(size=(n, d_z))

    logits = 0.6 * Z[:, 0] - 0.25 * Z[:, 1]
    e = 1.0 / (1.0 + np.exp(-logits))
    D = rng.binomial(1, e, size=n).astype(int)

    mu = 0.5 * Z[:, 0] - 0.2 * Z[:, 1] ** 2
    trend = 0.5 + 0.1 * Z[:, 0]  # common trend that depends on Z

    u = rng.normal(scale=1.0, size=n)  # unit fixed effect

    Y0 = mu + u + rng.normal(scale=1.0, size=n)
    Y1 = mu + trend + u + tau * D + rng.normal(scale=1.0, size=n)

    X = np.column_stack([D.astype(float), Z])
    return X, Y0, Y1, D

tau_true = 1.0

# Large population for an approximate truth
X_pop, Y0_pop, Y1_pop, D_pop = draw_panel(n=200_000, d_z=5, tau=tau_true, seed=1)

true_did = float(np.mean((Y1_pop - Y0_pop)[D_pop == 1] - (Y1_pop - Y0_pop)[D_pop == 0]))  # naive DID
# Our target here is "ATT on ΔY", whose true value equals tau_true by construction.
print("True tau (by construction):", tau_true)
print("Naive DID (difference in mean ΔY):", true_did)


## Estimate DID via grr_did

In [ ]:
# Sample a dataset from the same DGP
X, Y0, Y1, D = draw_panel(n=6000, d_z=5, tau=tau_true, seed=0)

psi = PolynomialBasis(degree=2, include_bias=True)
phi = TreatmentInteractionBasis(base_basis=psi)

gen = SquaredGenerator(C=0.0).as_generator()

res = grr_did(
    X=X,
    Y0=Y0,
    Y1=Y1,
    basis=phi,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res.summary_text())


## Compare to the true value

In [ ]:
for key, est in res.estimates.items():
    err = est.estimate - tau_true
    print(f"{key:>12s}: estimate={est.estimate: .6f},  error={err: .6f}")
